# Q-Learning

Q-learning is a **model-free reinforcement learning (RL)** algorithm used to find an optimal action-selection policy for finite **Markov Decision Processes (MDPs)**.

- **Model-free**: the agent does not need the environment transition model in advance.
- **Finite MDP**: best suited for environments with finite states and actions.
- **Q** stands for **Quality**: how good an action is in a given state.

## Core Concepts

1. **Value-based learning**: estimate the value of actions in states.
2. **Exploration vs. exploitation**: try new actions vs. use the best known action.
3. **State-action pairs**: learn values for `(state, action)` (not just states).
4. **Iterative improvement**: continuously update Q-values using experience.

## Problem Setup: Reach the Goal in a 3x3 Grid

We train an agent to move from a start tile to a goal tile while avoiding an obstacle.

- **Q-value**: expected utility of taking an action in a state.
- **Q-table**: lookup table storing all Q-values for every state-action pair.

In [ ]:
import numpy as np
import random
from typing import Tuple

# Define the grid world
GRID_SIZE = 3
START = (0, 0)
GOAL = (2, 2)
OBSTACLE = (1, 1)

# Actions: up, right, down, left
ACTIONS = [
    (-1, 0),
    (0, 1),
    (1, 0),
    (0, -1),
]

## Step 1: Environment Helper Functions

- `is_valid_state(state)`: checks boundaries and obstacle collision.
- `get_next_state(state, action)`: applies action, or stays in place if invalid.

In [ ]:
def is_valid_state(state: Tuple[int, int]) -> bool:
    return (0 <= state[0] < GRID_SIZE and
            0 <= state[1] < GRID_SIZE and
            state != OBSTACLE)


def get_next_state(state: Tuple[int, int], action: Tuple[int, int]) -> Tuple[int, int]:
    next_state = (state[0] + action[0], state[1] + action[1])
    return next_state if is_valid_state(next_state) else state

## Step 2: Q-Learning Hyperparameters

- `EPSILON = 0.3`: explore 30% of the time.
- `ALPHA = 0.3`: learning rate (how strongly new info updates old values).
- `GAMMA = 0.99`: discount factor for future rewards.
- `EPISODES = 10000`: total training runs.

In [ ]:
EPSILON = 0.3
ALPHA = 0.3
GAMMA = 0.99
EPISODES = 10000

## Step 3: Reward Design

Rewards guide behavior:

- `+100` for reaching the goal.
- `-10` for hitting walls/invalid moves (or obstacle attempt).
- `-1` per step to encourage short paths.

In [ ]:
def get_reward(state: Tuple[int, int], next_state: Tuple[int, int]) -> int:
    if next_state == GOAL:
        return 100
    elif next_state == OBSTACLE or next_state == state:
        return -10
    else:
        return -1

## Step 4: Action Selection (Exploration vs Exploitation)

Use an **epsilon-greedy** policy:

- with probability `EPSILON`, choose a random action (explore),
- otherwise choose action with max Q-value (exploit).

In [ ]:
def choose_action(state: Tuple[int, int], q_table: np.ndarray) -> Tuple[int, int]:
    if random.uniform(0, 1) < EPSILON:
        return random.choice(ACTIONS)
    return ACTIONS[int(np.argmax(q_table[state]))]

## Step 5: Update Rule (Bellman Equation)

Q-learning update:

`Q(s,a) <- Q(s,a) + alpha * [reward + gamma * max_a' Q(s',a') - Q(s,a)]`

- `ALPHA` controls update speed.
- `GAMMA` controls future reward importance.

In [ ]:
def update_q_table(
    q_table: np.ndarray,
    state: Tuple[int, int],
    action: Tuple[int, int],
    reward: int,
    next_state: Tuple[int, int],
) -> None:
    action_idx = ACTIONS.index(action)
    current_q = q_table[state][action_idx]
    target = reward + GAMMA * np.max(q_table[next_state])
    q_table[state][action_idx] = current_q + ALPHA * (target - current_q)

## Step 6: Training Loop

For each episode:

1. Start at `START`.
2. Choose action from current policy.
3. Move to next state and get reward.
4. Update Q-table.
5. Continue until reaching `GOAL`.

In [ ]:
def train_agent() -> np.ndarray:
    q_table = np.zeros((GRID_SIZE, GRID_SIZE, len(ACTIONS)))

    for _ in range(EPISODES):
        state = START
        while state != GOAL:
            action = choose_action(state, q_table)
            next_state = get_next_state(state, action)
            reward = get_reward(state, next_state)
            update_q_table(q_table, state, action, reward, next_state)
            state = next_state

    return q_table


q_table = train_agent()
q_table.shape

## Step 7: Visualization

We print:

- full Q-values per action in each state,
- best action and best Q-value per state.

In [ ]:
def visualize_q_table_as_grid(q_table: np.ndarray) -> None:
    action_symbols = ['^', '>', 'v', '<']

    print('\nDetailed Q-table Grid:')
    header = '   |' + '|'.join(f'   ({i},{j})   ' for i in range(GRID_SIZE) for j in range(GRID_SIZE)) + '|'
    print(header)
    print('-' * len(header))

    for action_idx, action_symbol in enumerate(action_symbols):
        row = f' {action_symbol} |'
        for i in range(GRID_SIZE):
            for j in range(GRID_SIZE):
                if (i, j) == GOAL:
                    cell = '   GOAL    '
                elif (i, j) == OBSTACLE:
                    cell = ' OBSTACLE  '
                else:
                    q_value = q_table[i, j, action_idx]
                    cell = f' {q_value:9.2f} '
                row += cell + '|'
        print(row)
        print('-' * len(header))


def visualize_best_actions_grid(q_table: np.ndarray) -> None:
    action_symbols = ['^', '>', 'v', '<']

    print('\nBest Actions Grid:')
    header = '-' * (14 * GRID_SIZE + 1)
    print(header)

    for i in range(GRID_SIZE):
        row = '| '
        for j in range(GRID_SIZE):
            if (i, j) == GOAL:
                cell = '   GOAL    '
            elif (i, j) == OBSTACLE:
                cell = ' OBSTACLE  '
            else:
                best_action_idx = int(np.argmax(q_table[i, j]))
                best_q_value = q_table[i, j, best_action_idx]
                cell = f'{action_symbols[best_action_idx]}:{best_q_value:7.2f}  '
            row += cell + ' | '
        print(row)
        print(header)

In [ ]:
visualize_q_table_as_grid(q_table)
visualize_best_actions_grid(q_table)

## Example Output (values may differ slightly)

Because exploration is random, your exact Q-values can vary per run, but the best-action pattern should converge to a shortest valid path to the goal.

## Key Takeaway

Q-tables work very well for small discrete environments.
For larger or continuous spaces, methods like **Deep Q-Networks (DQN)** approximate Q-values with neural networks.